# Tugas JS13 PBL - Klasifikasi Sleep Disorder

Notebook ini menggunakan dataset `Dataset/Sleep_health_and_lifestyle_dataset.csv`. Target klasifikasi adalah `Sleep Disorder` dengan tiga kelas: `None`, `Insomnia`, dan `Sleep Apnea`.

Eksperimen membandingkan dua hidden layer, jumlah neuron yang lebih besar, tambahan hidden layer, serta aktivasi Sigmoid dan ReLU.

In [4]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, log_loss
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

dataset_name = "Sleep_health_and_lifestyle_dataset.csv"
candidates = [
    Path("Dataset") / dataset_name,
    Path("../Dataset") / dataset_name,
    Path.cwd() / "Dataset" / dataset_name,
]
dataset_path = next((path for path in candidates if path.exists()), None)
if dataset_path is None:
    raise FileNotFoundError(f"Dataset tidak ditemukan. Lokasi yang dicek: {candidates}")

data = pd.read_csv(dataset_path)
data["Systolic"] = data["Blood Pressure"].str.split("/").str[0].astype(float)
data["Diastolic"] = data["Blood Pressure"].str.split("/").str[1].astype(float)
data = data.drop(columns=["Person ID", "Blood Pressure"])
data["Sleep Disorder"] = data["Sleep Disorder"].fillna("None")

X = data.drop(columns="Sleep Disorder")
y = data["Sleep Disorder"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Dataset: {dataset_path.resolve()}")
print(f"Jumlah data: {len(data)}")
print("Distribusi target:")
print(y.value_counts())

Dataset: C:\ListPc\244107020089_MachineLearning\Dataset\Sleep_health_and_lifestyle_dataset.csv
Jumlah data: 374
Distribusi target:
Sleep Disorder
None           219
Sleep Apnea     78
Insomnia        77
Name: count, dtype: int64


## Pra-pemrosesan dan model

Kolom numerik distandardisasi, sedangkan kolom kategori diubah menjadi one-hot encoding. Seluruh preprocessing dimasukkan ke dalam `Pipeline` agar tidak terjadi data leakage dari data testing.

In [5]:
numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X.select_dtypes(exclude=["number"]).columns.tolist()
preprocessor = ColumnTransformer([
    ("numeric", StandardScaler(), numeric_features),
    ("categorical", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_features),
])
X_train_preprocessed = preprocessor.fit_transform(X_train)
X_test_preprocessed = preprocessor.transform(X_test)
print("\nJumlah data setelah preprocessing:")
print(f"Total data: {len(X_train_preprocessed) + len(X_test_preprocessed)} baris")
print(f"Data training: {X_train_preprocessed.shape[0]} baris")
print(f"Data testing: {X_test_preprocessed.shape[0]} baris")
print(f"Jumlah fitur setelah encoding: {X_train_preprocessed.shape[1]} fitur")

def train_and_evaluate(name, hidden_layers, activation="relu"):
    model = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", MLPClassifier(
            hidden_layer_sizes=hidden_layers,
            activation=activation,
            solver="adam",
            batch_size=32,
            learning_rate_init=0.001,
            max_iter=500,
            early_stopping=True,
            validation_fraction=0.15,
            n_iter_no_change=25,
            random_state=42,
        )),
    ])
    start = time.perf_counter()
    model.fit(X_train, y_train)
    elapsed = time.perf_counter() - start
    predictions = model.predict(X_test)
    probabilities = model.predict_proba(X_test)
    return {
        "Konfigurasi": name,
        "Hidden layer": str(hidden_layers),
        "Aktivasi": activation,
        "Akurasi": accuracy_score(y_test, predictions),
        "Log loss": log_loss(y_test, probabilities, labels=model.classes_),
        "Waktu (detik)": elapsed,
        "model": model,
        "predictions": predictions,
    }


experiments = [
    train_and_evaluate("Baseline 2 hidden", (64, 32), "relu"),
    train_and_evaluate("Neuron diperbesar", (128, 64), "relu"),
    train_and_evaluate("Tambah hidden layer", (64, 32, 16), "relu"),
    train_and_evaluate("Sigmoid", (64, 32), "logistic"),
    train_and_evaluate("ReLU", (64, 32), "relu"),
]

results = pd.DataFrame([
    {key: value for key, value in item.items() if key not in {"model", "predictions"}}
    for item in experiments
])
display(results.style.format({
    "Akurasi": "{:.2%}",
    "Log loss": "{:.6f}",
    "Waktu (detik)": "{:.3f}",
}))


Jumlah data setelah preprocessing:
Total data: 374 baris
Data training: 299 baris
Data testing: 75 baris
Jumlah fitur setelah encoding: 26 fitur


,Konfigurasi,Hidden layer,Aktivasi,Akurasi,Log loss,Waktu (detik)
0,Baseline 2 hidden,"(64, 32)",relu,93.33%,0.322520,0.217
1,Neuron diperbesar,"(128, 64)",relu,94.67%,0.211061,0.320
2,Tambah hidden layer,"(64, 32, 16)",relu,94.67%,0.224762,0.320
3,Sigmoid,"(64, 32)",logistic,94.67%,0.446372,0.309
4,ReLU,"(64, 32)",relu,93.33%,0.322520,0.186


In [6]:
best_index = results["Akurasi"].idxmax()
best = experiments[best_index]
print("Model terbaik:", results.loc[best_index, "Konfigurasi"])
print(f"Akurasi: {results.loc[best_index, 'Akurasi']:.4f}")
print("\nClassification report:")
print(classification_report(y_test, best["predictions"], zero_division=0))
print("Confusion matrix:")
print(pd.DataFrame(
    confusion_matrix(y_test, best["predictions"], labels=sorted(y.unique())),
    index=sorted(y.unique()),
    columns=sorted(y.unique()),
))

Model terbaik: Neuron diperbesar
Akurasi: 0.9467

Classification report:
              precision    recall  f1-score   support

    Insomnia       0.87      0.87      0.87        15
        None       1.00      0.98      0.99        44
 Sleep Apnea       0.88      0.94      0.91        16

    accuracy                           0.95        75
   macro avg       0.92      0.93      0.92        75
weighted avg       0.95      0.95      0.95        75

Confusion matrix:
             Insomnia  None  Sleep Apnea
Insomnia           13     0            2
None                1    43            0
Sleep Apnea         1     0           15


## Kesimpulan

Model dievaluasi berdasarkan akurasi, log loss, dan waktu pelatihan. Penambahan neuron atau hidden layer meningkatkan kapasitas model, tetapi dapat menambah waktu training. Aktivasi ReLU dan Sigmoid dibandingkan pada dataset yang sama dengan pembagian data dan random state yang sama.